In [2]:
from pathlib import Path
import tensorflow as tf
import sys

# appending llm_components path to sys.path to easily import
axiom_utils = Path('/kaggle/input/datasets/harshit1234g/axiomlm-utils')
sys.path.append(str(axiom_utils))
import llm_components as lc

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

## Paths

In [4]:
dolly_dir = axiom_utils / 'dolly_15k'
features_path = str(dolly_dir / 'processed_features.npy')
labels_path = str(dolly_dir / 'processed_labels.npy')
tokenizer_path = str(axiom_utils / 'sp_tokenizer.model')

# I uploaded the model on kaggle, so it has a different path
model_path = Path('/kaggle/input/models/harshit1234g/axiomlm/tensorflow2/default/4/AxiomLM-33M-Base.keras')

## Loading data

In [5]:
tokenizer = lc.load_sp_tokenizer(tokenizer_path)

In [6]:
full_ds = lc.load_sft_dataset(
    features_path,
    labels_path,
    pad_token_id= tokenizer.pad_id(),
    batch_size= 64,
    shuffle_buffer= 10_000
)

I0000 00:00:1772635918.568707      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772635918.574603      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [7]:
count = 0
for _ in full_ds.as_numpy_iterator():
    count += 1

In [8]:
train_size = int(0.8 * count)
val_size = int(0.1 * count)
test_size = count - train_size - val_size

In [9]:
print(f'Total batches: {count}')
print(f'{train_size = }, {val_size = }, {test_size = }')

Total batches: 216
train_size = 172, val_size = 21, test_size = 23


In [10]:
train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size).take(val_size)
test_ds = full_ds.skip(train_size + val_size)

## SFT

In [11]:
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


In [12]:
with strategy.scope():
    model = tf.keras.models.load_model(model_path)
    
    # freezing the initial embedding and transformer layers
    for layer in model.layers[:6]:
        layer.trainable = False

    for layer in model.layers[6:]:
        layer.trainable = True
    
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate= 1e-5,
        weight_decay= 0.0,   # removing regularization, so that model could adapt the new behaviour easily
        beta_2= 0.98,
        clipnorm= 1.0
    )

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits= True,
        ignore_class= -100
    )

    model.compile(
        optimizer= optimizer,
        loss= loss_fn
    )

In [13]:
model.summary()

Model: "gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 512, 512)          │     8,192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (512, 512)             │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_16          │ ?                      │         1,024 │
│ (LayerNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,661,952 (128.41 MB)

 Trainable params: 12,604,416 (48.08 MB)

 Non-trainable params: 21,057,536 (80.33 MB)

In [14]:
history = model.fit(
    train_ds,
    epochs= 2,
    validation_data= val_ds
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


172/172 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - loss: 0.0327 - ppl: nan - val_loss: 0.0000e+00 - val_ppl: nan
Epoch 2/2
172/172 ━━━━━━━━━━━━━━━━━━━━ 224s 1s/step - loss: 0.0752 - ppl: nan - val_loss: 0.1093 - val_ppl: nan


In [15]:
test_loss, test_ppl = model.evaluate(test_ds)
print(f'{test_loss = }\n{test_ppl = }')

23/23 ━━━━━━━━━━━━━━━━━━━━ 20s 722ms/step - loss: 0.0380 - ppl: nan
test_loss = 0.10051995515823364
test_ppl = nan


In [16]:
model.save('AxiomLM-33M-Instruct.keras')